In [1]:
import os
os.chdir("../")
%pwd

'C:\\Users\\Sayantan Nandi\\Desktop\\mlops\\developer-burnout-project'

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataTransformationConfig:
    root_dir: Path
    data_path: Path

In [3]:
from src.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from src.utils.common import read_yaml, create_directories

In [4]:
class ConfigurationManager:
    def __init__(self,
                 config_path=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH,
                ):
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path
        )

        return data_transformation_config

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [18]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        create_directories([self.config.root_dir])

    def transform_and_save(self):
        data = pd.read_csv(self.config.data_path)

        data = data.dropna(subset=["burnout_level"])
        num_cols = data.select_dtypes(include='number')
        data[num_cols.columns] = num_cols.fillna(num_cols.mean())

        train_df, test_df = train_test_split(
            data,
            test_size=0.2,
            random_state=42,
            stratify=data["burnout_level"]
        )

        train_df.to_csv(f"{self.config.root_dir}/train.csv", index=False)
        test_df.to_csv(f"{self.config.root_dir}/test.csv", index=False)



In [19]:
configurationManager = ConfigurationManager()
data_transformation_config = configurationManager.get_data_transformation_config()
data_transformation = DataTransformation(config=data_transformation_config)
data_transformation.transform_and_save()

[2026-04-16 03:00:49,076: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-16 03:00:49,077: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-16 03:00:49,078: INFO: common: created directory at: artifacts]
[2026-04-16 03:00:49,079: INFO: common: created directory at: artifacts/data_transformation]
[2026-04-16 03:00:49,080: INFO: common: created directory at: artifacts/data_transformation]
